# Benchmark audité du corpus GeNIS 2025 — pipeline expérimental

**Article :** *A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus*
— Ala Bahri, Farah Jemili, Mohamed Mosbah.

---

## Conçu pour Colab gratuit (12 Go de RAM, GPU intermittent)

Trois mécanismes rendent l'exécution robuste aux déconnexions :

1. **Persistance immédiate.** Chaque modèle entraîné écrit ses métriques dans
   `article1_results.json` et ses probabilités dans `probs/` **sur Drive**, juste après
   son calcul. Une déconnexion ne fait perdre que le modèle en cours.
2. **Inventaire au démarrage (§1.3).** Le notebook affiche exactement ce qui est déjà
   calculé et ce qui reste — vous n'avez jamais à deviner.
3. **Exécution par lots (§5).** Chaque lot est une cellule indépendante que vous pouvez
   lancer seule : *modèles rapides* (CPU suffisant) et *modèles profonds* (GPU recommandé)
   sont séparés, pour s'adapter à la disponibilité du GPU.

**Économie de mémoire** : le tableau brut (338 820 × 125, ~2 Go) est converti en matrice
`float32` puis **libéré explicitement** ; les données prétraitées sont mises en cache sur
Drive, si bien qu'un redémarrage coûte ~1 min au lieu de ~15. Les matrices normalisées
sont calculées **une fois par (condition, split)** et partagées par tous les modèles du lot.

**Affichage de progression** : chaque modèle annonce son démarrage, la RAM utilisée, le
temps écoulé et le temps restant estimé ; les modèles profonds affichent chaque époque.

---

## Marche à suivre

| Situation | Cellules à exécuter |
|---|---|
| Première fois | §1 → §7 dans l'ordre (*Tout exécuter*) |
| Après une déconnexion | §1 (setup, ~2 min) puis **le lot §5 interrompu** |
| Session sans GPU | §1, puis lots « rapides » §5.1a / §5.3a / §5.5a |
| Session avec GPU | §1, puis lots « profonds » §5.1b / §5.3b / §5.5b |
| Tout est calculé | §1 puis §6 et §7 (figures, tables, export) |

## Protocole (gelé)

- Scaler ajusté sur le train seul, distribution naturelle, aucune rééquilibration du test.
- Conditions de features : `full` (avec positionnelles) → `clean` (sans) → `audited` (après liste noire, §5.2).
- Splits : stratifié 60/20/20 × 5 graines, et **temporel par classe**.
- Bras d'hyperparamètres : `défaut` (architectures BAg-IDS publiées) et `réglé` (recherche aléatoire à budget déclaré).


## 1. Configuration, données et inventaire

In [ ]:
# 1.1 — Environnement et constantes
import os, sys, glob, json, time, gc, math, shutil, pathlib, platform, itertools, warnings, pickle, io
import numpy as np, pandas as pd, sklearn, scipy, joblib
import tensorflow as tf
import matplotlib, matplotlib.pyplot as plt
from IPython.display import display
try:
    import xgboost, lightgbm, psutil
except ImportError:
    %pip -q install xgboost lightgbm psutil
    import xgboost, lightgbm, psutil

warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})

# ---------------- protocole (gele) ----------------
SEEDS        = [1, 2, 3, 4, 5]
SPLIT_FRAC   = (0.60, 0.20, 0.20)
WINDOW_GAP_S = 1800
BOOTSTRAP_B  = 1000
ECE_BINS     = 15

# ---------------- interrupteurs ----------------
RUN_HPO        = True     # recherche d'hyperparametres + bras regle
RUN_INTERVALS  = True     # etude 5/10/30 s
RUN_COST       = True     # banc de cout CPU
SAVE_MODELS    = True     # persistance des modeles de reference
KNN_MAX_TRAIN  = 50000    # budget memoire k-NN (declare dans le papier)

def mem():                                  # RAM du processus, en Go
    return psutil.Process().memory_info().rss / 1e9
def mem_total():
    return psutil.virtual_memory().total / 1e9
def fmt(sec):
    sec = int(sec)
    return f"{sec//3600} h {sec%3600//60:02d}" if sec >= 3600 else (
           f"{sec//60} min {sec%60:02d}" if sec >= 60 else f"{sec} s")

GPU = bool(tf.config.list_physical_devices("GPU"))
print(f"python {platform.python_version()} | tensorflow {tf.__version__} | sklearn {sklearn.__version__}")
print(f"xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}")
print(f"RAM : {mem():.1f} / {mem_total():.1f} Go utilises")
print("GPU :", "OUI" if GPU else "NON — lancez plutot les lots 'rapides' (§5.1a/§5.3a/§5.5a) ; "
      "les modeles profonds sur CPU prennent des heures")


In [ ]:
# 1.2 — Drive, dossiers, reprise
from google.colab import drive
drive.mount('/content/drive')

WORK = pathlib.Path("/content/genis"); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SAVE   = pathlib.Path("/content/drive/MyDrive/GeNIS/article1_final")
PROBS  = SAVE / "probs"; FIGS = SAVE / "figures"; SUPP = SAVE / "figures_annexe"
TABS   = SAVE / "tables"; MODELS = SAVE / "models"; CACHE = SAVE / "cache"
for p in (SAVE, PROBS, FIGS, SUPP, TABS, MODELS, CACHE): p.mkdir(parents=True, exist_ok=True)

RES_PATH = SAVE / "article1_results.json"
RESULTS  = json.loads(RES_PATH.read_text()) if RES_PATH.exists() else {}
for k in ("models", "meta", "history"): RESULTS.setdefault(k, {})
RESULTS["meta"].update({
    "updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "protocol": {"seeds": SEEDS, "split": SPLIT_FRAC, "ece_bins": ECE_BINS,
                 "bootstrap": BOOTSTRAP_B, "knn_max_train": KNN_MAX_TRAIN},
    "env": {"python": platform.python_version(), "tensorflow": tf.__version__,
            "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
            "lightgbm": lightgbm.__version__, "gpu": GPU}})
def save_results():
    tmp = RES_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RESULTS, indent=1, default=float), encoding="utf-8")
    tmp.replace(RES_PATH)          # ecriture atomique : jamais de JSON tronque
save_results()
print(f"sortie : {SAVE}")
print(f"runs deja enregistres sur Drive : {len(RESULTS['models'])}")

# ---- corpus ----
drive_zip = "/content/drive/MyDrive/GeNIS/2-flows.zip"
if not pathlib.Path("flows").exists():
    if pathlib.Path(drive_zip).exists():
        print("copie de 2-flows.zip depuis Drive…"); shutil.copy(drive_zip, "2-flows.zip")
    else:
        print("telechargement depuis Zenodo (~380 Mo)…")
        !wget -q --show-progress "https://zenodo.org/records/14919237/files/2-flows.zip?download=1" -O 2-flows.zip
    !unzip -o -q 2-flows.zip -d flows
csvs = sorted(glob.glob("flows/**/*.csv", recursive=True))
assert csvs, "aucun CSV trouve"
print(f"{len(csvs)} fichiers CSV")


In [ ]:
# 1.3 — Prétraitement mis en cache : chargement rapide au redemarrage
INTERVALS  = ["5", "10", "30", "60"]
IDENT_LIST = ["FlowID", "AutoId", "SrcAddr", "DstAddr", "Ssaddr", "Sdaddr",
              "SrcMac", "DstMac", "SrcOui", "DstOui", "Sport", "Dport",
              "sIpId", "dIpId", "sMpls", "dMpls", "sAS", "dAS", "iAS",
              "sCo", "dCo", "sVid", "dVid"]
POS_LIST   = ["StartTime", "LastTime", "Rank", "Seq"]
LAB_LIST   = ["BinaryLabel", "CategoryLabel", "SubCategoryLabel"]

def feature_sets(d):
    ident = [c for c in IDENT_LIST if c in d.columns]
    pos   = [c for c in POS_LIST if c in d.columns]
    num = (d.drop(columns=ident + LAB_LIST, errors="ignore")
             .select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan))
    nu = num.nunique(dropna=True); const = nu[nu <= 1].index.tolist()
    num = num.drop(columns=const).fillna(0.0).astype(np.float32)
    return num, list(num.columns), [c for c in num.columns if c not in pos], pos, ident, const

def load_slice_df(iv):
    files = [c for c in csvs if f"flows-{iv}-sec" in c]
    d = pd.concat([pd.read_csv(c, low_memory=False) for c in files], ignore_index=True)
    ysub = d["SubCategoryLabel"].astype(str).str.strip()
    y9 = ysub.where(~ysub.str.startswith("benign"), "benign")   # 3 profils benins fusionnes
    t = pd.to_numeric(d["StartTime"], errors="coerce")
    assert t.notna().all(), "StartTime non numerique"
    return d, y9, t

CACHE_NPZ = CACHE / "slice60.npz"; CACHE_META = CACHE / "slice60_meta.json"
if CACHE_NPZ.exists() and CACHE_META.exists():
    z = np.load(CACHE_NPZ); meta = json.loads(CACHE_META.read_text())
    X_ALL, y, t_np, POS_RAW = z["X"], z["y"].astype(int), z["t"], z["pos"]
    FEAT_ALL   = meta["feat_all"];   F_FULL  = meta["f_full"]
    F_CLEAN    = meta["f_clean"];    POSITIONAL = meta["positional"]
    IDENTIFIERS = meta["identifiers"]; DROPPED_CONST = meta["const"]
    CLASS_NAMES = meta["classes"];   CLASS_COUNTS = meta["counts"]
    print(f"cache charge en quelques secondes ({CACHE_NPZ.name})")
else:
    print("premier chargement (parsing des CSV, ~5-10 min)…")
    df, y9_raw, t_start = load_slice_df("60")
    num, F_FULL, F_CLEAN, POSITIONAL, IDENTIFIERS, DROPPED_CONST = feature_sets(df)
    from sklearn.preprocessing import LabelEncoder
    le_ = LabelEncoder(); y = le_.fit_transform(y9_raw)
    CLASS_NAMES  = list(le_.classes_)
    CLASS_COUNTS = pd.Series(y9_raw).value_counts().to_dict()
    FEAT_ALL = list(num.columns)
    X_ALL    = np.ascontiguousarray(num.values, dtype=np.float32)
    t_np     = t_start.values.astype(np.float64)
    POS_RAW  = np.column_stack([pd.to_numeric(df[c], errors="coerce").fillna(0).values
                                for c in POSITIONAL]).astype(np.float64)
    np.savez_compressed(CACHE_NPZ, X=X_ALL, y=y.astype(np.int16), t=t_np, pos=POS_RAW)
    CACHE_META.write_text(json.dumps(
        {"feat_all": FEAT_ALL, "f_full": F_FULL, "f_clean": F_CLEAN,
         "positional": POSITIONAL, "identifiers": IDENTIFIERS, "const": DROPPED_CONST,
         "classes": CLASS_NAMES, "counts": CLASS_COUNTS}, indent=1), encoding="utf-8")
    # LIBERATION EXPLICITE : le tableau brut pese ~2 Go et ne sert plus
    del df, num, y9_raw, t_start; gc.collect()
    print("cache ecrit ; tableau brut libere")

C = len(CLASS_NAMES); BENIGN_IDX = CLASS_NAMES.index("benign")
IDX = {n: i for i, n in enumerate(FEAT_ALL)}
def cols_idx(names): return np.array([IDX[n] for n in names], dtype=int)
POS_IDX = {n: i for i, n in enumerate(POSITIONAL)}
print(f"{len(y):,} flux | {C} classes | features full {len(F_FULL)} / clean {len(F_CLEAN)}")
print(f"RAM apres chargement : {mem():.1f} / {mem_total():.1f} Go")


In [ ]:
# 1.4 — INVENTAIRE : ce qui est deja calcule, ce qui reste
FAST_NAMES = ["majority", "logreg", "nb", "knn", "rf", "xgboost", "lightgbm"]
DEEP_NAMES = ["rnn", "cnn", "dnn", "ftt"]
TUNABLE    = [m for m in FAST_NAMES if m != "majority"] + DEEP_NAMES
STRATS       = [f"strat_seed{s}" for s in SEEDS]
ILLUS_SPLITS = ["strat_seed1", "temporal"]

def run_key(m, tag, sk_, arm):
    return f"{m}{'' if arm == 'default' else '#tuned'}|{tag}|{sk_}"
def kfile(key): return PROBS / f"{key.replace('|', '_').replace('#', '-')}.npz"
def done(key):  return key in RESULTS["models"] and kfile(key).exists()

def plan_runs():
    P = []
    for tag in ("full", "clean"):                       # illustration avant audit
        for sk_ in ILLUS_SPLITS:
            for m in FAST_NAMES: P.append((m, tag, sk_, "default"))
        for m in DEEP_NAMES:     P.append((m, tag, "strat_seed1", "default"))
    for sk_ in STRATS + ["temporal"]:                   # tableau principal
        for m in FAST_NAMES:
            if m == "knn" and sk_ not in ("strat_seed1", "temporal"): continue
            P.append((m, "audited", sk_, "default"))
        for m in DEEP_NAMES:     P.append((m, "audited", sk_, "default"))
    if RUN_HPO:
        for sk_ in STRATS + ["temporal"]:               # bras regle
            for m in TUNABLE:
                if m == "knn" and sk_ not in ("strat_seed1", "temporal"): continue
                P.append((m, "audited", sk_, "tuned"))
    return P

def inventory():
    P = plan_runs()
    grp = {}
    for m, tag, sk_, arm in P:
        fam = "profond" if m in DEEP_NAMES else "rapide"
        k = (f"{tag}/{arm}", fam)
        d, t_ = grp.get(k, (0, 0))
        grp[k] = (d + (1 if done(run_key(m, tag, sk_, arm)) else 0), t_ + 1)
    rows = [{"lot": k[0], "famille": k[1], "faits": v[0], "total": v[1],
             "etat": "termine" if v[0] == v[1] else ("a faire" if v[0] == 0 else "en cours")}
            for k, v in sorted(grp.items())]
    inv = pd.DataFrame(rows)
    display(inv)
    nd = sum(1 for r in P if done(run_key(*r)))
    print(f"TOTAL : {nd}/{len(P)} runs ({nd/len(P):.0%})")
    for lab, cond in [("audit / liste noire", "audit" in RESULTS),
                      ("recherche d'hyperparametres", "hpo" in RESULTS),
                      ("autoencodeur", "autoencoder" in RESULTS),
                      ("multi-intervalles", bool(RESULTS.get("intervals"))),
                      ("banc de cout", "cost" in RESULTS),
                      ("calibration", "calibration" in RESULTS),
                      ("statistiques", "stats" in RESULTS)]:
        print(f"  {lab:32s} {'OK' if cond else '- a faire'}")
    return inv

_ = inventory()


## 2. Exploration des données (EDA)

Section légère et entièrement mise en cache : au redémarrage, elle se réexécute en quelques
secondes. Si toutes les figures existent déjà, elle les régénère à l'identique.


In [ ]:
# 2.1 — Volumes et desequilibre par intervalle (Table 1)
if "interval_stats" not in RESULTS:
    st = {}
    for iv in INTERVALS:
        counts = {pathlib.Path(f).stem: sum(1 for _ in open(f, "rb")) - 1
                  for f in csvs if f"flows-{iv}-sec" in f}
        tot = sum(counts.values()); ben = sum(v for k, v in counts.items() if k.startswith("benign"))
        st[iv] = {"files": counts, "total": tot, "benign": ben, "benign_share": ben / tot}
    RESULTS["interval_stats"] = st; save_results()

t1 = pd.DataFrame({iv: {"flux total": RESULTS["interval_stats"][iv]["total"],
                        "flux benins": RESULTS["interval_stats"][iv]["benign"],
                        "part benin (%)": round(RESULTS["interval_stats"][iv]["benign_share"] * 100, 2)}
                   for iv in INTERVALS}).T
t1.index.name = "intervalle (s)"; display(t1)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(INTERVALS, [RESULTS["interval_stats"][iv]["total"] for iv in INTERVALS], color="#4C72B0")
ax[0].set_title("Nombre de flux"); ax[0].set_xlabel("intervalle (s)")
ax[1].bar(INTERVALS, [RESULTS["interval_stats"][iv]["benign_share"] * 100 for iv in INTERVALS], color="#55A868")
ax[1].set_title("Part de trafic benin (%)"); ax[1].set_xlabel("intervalle (s)")
plt.savefig(SUPP / "A1_intervals.png"); plt.savefig(SUPP / "A1_intervals.pdf"); plt.show()


In [ ]:
# 2.2 — Distribution des classes (A2) et chronologie (Figure 1)
dist = pd.DataFrame({"flux": pd.Series(CLASS_COUNTS),
                     "part (%)": (pd.Series(CLASS_COUNTS) / len(y) * 100).round(2)}
                    ).sort_values("flux", ascending=False)
display(dist)
print(f"desequilibre max/min : {dist['flux'].max() / dist['flux'].min():.1f}:1")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
dist["flux"].plot.bar(ax=ax[0], color=["#55A868" if c == "benign" else "#C44E52" for c in dist.index])
ax[0].set_yscale("log"); ax[0].set_ylabel("flux (log)")
ax[0].set_title("Distribution naturelle des 9 classes")
ax[1].pie([int((y != BENIGN_IDX).sum()), int((y == BENIGN_IDX).sum())],
          labels=["attaque", "benin"], autopct="%1.1f%%", colors=["#C44E52", "#55A868"])
ax[1].set_title("Repartition binaire")
plt.savefig(SUPP / "A2_class_distribution.png"); plt.savefig(SUPP / "A2_class_distribution.pdf"); plt.show()

def activity_windows(times, gap=WINDOW_GAP_S):
    ts = np.sort(times); brk = np.where(np.diff(ts) > gap)[0]
    return [(s[0], s[-1], len(s)) for s in np.split(ts, brk + 1)]
t0 = t_np.min()
rows = []
for i, cn in enumerate(CLASS_NAMES):
    w = activity_windows(t_np[y == i])
    rows.append({"classe": cn, "flux": int((y == i).sum()), "fenetres": len(w),
                 "duree active (h)": round(sum(e - s for s, e, _ in w) / 3600, 2),
                 "debut (h)": round((w[0][0] - t0) / 3600, 1),
                 "fin (h)": round((w[-1][1] - t0) / 3600, 1)})
twin = pd.DataFrame(rows).set_index("classe"); display(twin)
RESULTS["activity_windows"] = twin.reset_index().to_dict("records"); save_results()

fig, ax = plt.subplots(figsize=(9, 3.4))
rng = np.random.RandomState(0)
for i, cn in enumerate(CLASS_NAMES):
    ix = np.where(y == i)[0]; ix = rng.choice(ix, size=min(3000, len(ix)), replace=False)
    ax.plot((t_np[ix] - t0) / 3600, np.full(len(ix), i), "|", ms=5,
            color="#55A868" if i == BENIGN_IDX else "#C44E52", alpha=.3)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("heures depuis le debut de la capture (6-12 fevrier 2025)")
ax.set_title("Figure 1 — fenetres temporelles par classe : etroites et disjointes")
plt.savefig(FIGS / "fig1_timeline.png"); plt.savefig(FIGS / "fig1_timeline.pdf"); plt.show()


In [ ]:
# 2.3 — Correlation (A3) et projection PCA (A4)
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

ci = cols_idx(F_CLEAN)
sub = X_ALL[np.random.RandomState(0).choice(len(y), min(20000, len(y)), replace=False)][:, ci]
corr = np.corrcoef(sub, rowvar=False)
hi = int((np.abs(np.triu(corr, 1)) > .95).sum())
print(f"paires de features |r| > 0.95 : {hi}")
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title(f"Correlation des {len(F_CLEAN)} features comportementales")
ax.set_xticks([]); ax.set_yticks([]); plt.colorbar(im, label="Pearson")
plt.savefig(SUPP / "A3_feature_correlation.png"); plt.savefig(SUPP / "A3_feature_correlation.pdf"); plt.show()

rng = np.random.RandomState(0)
ix = np.concatenate([rng.choice(np.where(y == c)[0], min(1500, int((y == c).sum())), replace=False)
                     for c in range(C)])
Xs = np.nan_to_num(RobustScaler().fit_transform(X_ALL[ix][:, ci]), nan=0., posinf=0., neginf=0.)
pc = PCA(n_components=2, random_state=0).fit(Xs); Z = pc.transform(Xs)
fig, ax = plt.subplots(figsize=(6.5, 5)); cmap = plt.get_cmap("tab10")
for c in range(C):
    m = y[ix] == c
    ax.scatter(Z[m, 0], Z[m, 1], s=4, alpha=.45, label=CLASS_NAMES[c],
               color="#55A868" if c == BENIGN_IDX else cmap(c % 10))
ax.set_xlabel(f"CP1 ({pc.explained_variance_ratio_[0]*100:.1f} %)")
ax.set_ylabel(f"CP2 ({pc.explained_variance_ratio_[1]*100:.1f} %)")
ax.set_title("Projection PCA des features comportementales"); ax.legend(fontsize=7, markerscale=2, ncol=2)
plt.savefig(SUPP / "A4_pca.png"); plt.savefig(SUPP / "A4_pca.pdf"); plt.show()
RESULTS["slice60"] = {"n": int(len(y)), "classes": CLASS_NAMES, "class_counts": CLASS_COUNTS,
                      "benign_share": float((y == BENIGN_IDX).mean()),
                      "features_full": F_FULL, "features_clean": F_CLEAN,
                      "positional": POSITIONAL, "identifiers_excluded": IDENTIFIERS,
                      "constant_dropped": DROPPED_CONST, "high_corr_pairs": hi}
del sub, Xs, Z; gc.collect(); save_results()


## 3. Splits et sondes de raccourci

**Catégorisation des colonnes** (listes nominatives, jamais de motifs de chaînes) :
identifiants (IP, MAC, ports, n° de flux) exclus partout ; positionnelles (`StartTime`,
`LastTime`, `Rank`, `Seq`) dans la condition `full` uniquement ; comportementales
candidates, l'audit §5.2 tranche.


In [ ]:
# 3.1 — Splits geles : stratifie x5 + temporel PAR CLASSE
from sklearn.model_selection import train_test_split

def temporal_split_per_class(y_, t_, frac=SPLIT_FRAC):
    tr, va, te = [], [], []
    for c in range(C):
        ix = np.where(y_ == c)[0]; ix = ix[np.argsort(t_[ix], kind="stable")]
        n = len(ix); a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
        tr.append(ix[:a]); va.append(ix[a:b]); te.append(ix[b:])
    return np.concatenate(tr), np.concatenate(va), np.concatenate(te)

SPLIT_KEYS  = STRATS + ["temporal"]
SPLITS_PATH = SAVE / "frozen_splits_60s.npz"
if SPLITS_PATH.exists():
    z = np.load(SPLITS_PATH)
    splits = {k: (z[f"{k}_train"], z[f"{k}_val"], z[f"{k}_test"]) for k in SPLIT_KEYS}
    print("splits recharges depuis le gel")
else:
    idx = np.arange(len(y)); splits = {}
    for s in SEEDS:
        itr, itmp = train_test_split(idx, test_size=.40, random_state=s, stratify=y)
        iva, ite  = train_test_split(itmp, test_size=.50, random_state=s, stratify=y[itmp])
        splits[f"strat_seed{s}"] = (itr, iva, ite)
    splits["temporal"] = temporal_split_per_class(y, t_np)
    np.savez_compressed(SPLITS_PATH, **{f"{k}_{p}": v for k, (a, b, c_) in splits.items()
                                        for p, v in zip(("train", "val", "test"), (a, b, c_))})
    print(f"splits generes et geles -> {SPLITS_PATH.name}")

tr, va, te = splits["temporal"]
chk = pd.DataFrame({p: pd.Series(y[i_]).value_counts().reindex(range(C), fill_value=0).values
                    for p, i_ in zip(("train", "val", "test"), (tr, va, te))}, index=CLASS_NAMES)
display(chk)
ok = all(t_np[tr][y[tr] == c].max() <= t_np[te][y[te] == c].min()
         for c in range(C) if (y[tr] == c).any() and (y[te] == c).any())
print("invariant temporel (test posterieur au train, par classe) :", "OK" if ok else "VIOLE")
print("toutes les classes presentes partout :", bool((chk > 0).all().all()))
RESULTS["temporal_class_table"] = chk.to_dict(); save_results()


In [ ]:
# 3.2 — Sondes de raccourci + diagnostic du split chronologique global
from sklearn.tree import DecisionTreeClassifier

def probe_matrix(names):
    # Matrice float64 : les positionnelles viennent de POS_RAW (precision preservee)
    cols = []
    for n in names:
        cols.append(POS_RAW[:, POS_IDX[n]] if n in POS_IDX else X_ALL[:, IDX[n]].astype(np.float64))
    return np.column_stack(cols)

def probe(names, key):
    tr_, _, te_ = splits[key]; Xp = probe_matrix(names)
    clf = DecisionTreeClassifier(random_state=1).fit(Xp[tr_], y[tr_])
    acc = float((clf.predict(Xp[te_]) == y[te_]).mean())
    del Xp, clf; gc.collect(); return acc

if "shortcut_probes" not in RESULTS:
    pr = {"chance_majority": float(pd.Series(y).value_counts(normalize=True).max())}
    for nm, cols in [("starttime_only", ["StartTime"]), ("positional_all", POSITIONAL)]:
        accs = [probe(cols, f"strat_seed{s}") for s in SEEDS]
        pr[nm] = {"strat_mean": float(np.mean(accs)), "strat_std": float(np.std(accs)),
                  "temporal": probe(cols, "temporal")}
    RESULTS["shortcut_probes"] = pr; save_results()
pr = RESULTS["shortcut_probes"]
print(f"hasard (classe majoritaire) : {pr['chance_majority']:.4f}")
for nm, lab in [("starttime_only", "StartTime seul"), ("positional_all", "positionnelles")]:
    print(f"{lab:16s} : stratifie {pr[nm]['strat_mean']:.4f} (+/- {pr[nm]['strat_std']:.4f})"
          f" | temporel {pr[nm]['temporal']:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
labs = ["StartTime seul", "positionnelles"]; xs = np.arange(2); w = .35
ax.bar(xs - w/2, [pr[k]["strat_mean"] for k in ("starttime_only", "positional_all")], w,
       yerr=[pr[k]["strat_std"] for k in ("starttime_only", "positional_all")],
       label="split stratifie", color="#C44E52", capsize=3)
ax.bar(xs + w/2, [pr[k]["temporal"] for k in ("starttime_only", "positional_all")], w,
       label="split temporel", color="#4C72B0")
ax.axhline(pr["chance_majority"], ls="--", c="k", lw=.8, label="hasard")
ax.set_xticks(xs); ax.set_xticklabels(labs); ax.set_ylabel("accuracy 9 classes")
ax.set_title("Sondes de raccourci temporel"); ax.legend(fontsize=8)
plt.savefig(SUPP / "A5_shortcut_probes.png"); plt.savefig(SUPP / "A5_shortcut_probes.pdf"); plt.show()

# diagnostic : le split chronologique GLOBAL est degenere (resultat, pas protocole)
o = np.argsort(t_np, kind="stable"); n = len(o)
gtab = pd.DataFrame({"train (60 % premiers)": pd.Series(y[o[:int(.6*n)]]).value_counts().reindex(range(C), fill_value=0).values,
                     "test (20 % derniers)":  pd.Series(y[o[int(.8*n):]]).value_counts().reindex(range(C), fill_value=0).values},
                    index=CLASS_NAMES)
display(gtab)
missing = [CLASS_NAMES[c] for c in range(C) if gtab.iloc[c, 1] > 0 and gtab.iloc[c, 0] == 0]
print(f"classes du test jamais vues a l'entrainement : {missing or 'aucune'}")
RESULTS["global_chrono_table"] = gtab.to_dict(); save_results()


## 4. Modèles et moteur d'exécution

Douze modèles supervisés et un autoencodeur. Le trio RNN/CNN/DNN reprend **exactement**
les architectures publiées dans BAg-IDS (provenance) ; le bras réglé (§5.4) explore autour
sans changer la famille.

**Exclusions déclarées** : SVM à noyau (entraînement en O(n²), inentraînable à budget égal)
et le trio k-means/EM/SOM (remplacé par l'autoencodeur). **Budget k-NN** : entraînement sur
au plus 50 000 flux échantillonnés de façon stratifiée, pour borner la mémoire — déclaré
dans le papier.


In [ ]:
# 4.1 — Definitions (parametrees : les valeurs par defaut sont le bras "defaut")
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                             roc_auc_score, confusion_matrix, roc_curve,
                             precision_recall_curve, classification_report)

DEFAULTS_SK = {"majority": {}, "logreg": {"max_iter": 1000}, "nb": {},
               "knn": {"n_neighbors": 5},
               "rf": {"n_estimators": 200},
               "xgboost": {"n_estimators": 300, "max_depth": 8, "learning_rate": .1},
               "lightgbm": {"n_estimators": 300, "num_leaves": 63, "learning_rate": .1}}
def make_sk(name, params=None):
    p = {**DEFAULTS_SK[name], **(params or {})}
    if name == "majority": return DummyClassifier(strategy="most_frequent")
    if name == "logreg":   return LogisticRegression(n_jobs=-1, **p)
    if name == "nb":       return GaussianNB(**p)
    if name == "knn":      return KNeighborsClassifier(n_jobs=2, **p)
    if name == "rf":       return RandomForestClassifier(n_jobs=2, random_state=0, **p)
    if name == "xgboost":  return XGBClassifier(tree_method="hist", n_jobs=2, random_state=0,
                                                eval_metric="mlogloss", **p)
    if name == "lightgbm": return LGBMClassifier(n_jobs=2, random_state=0, verbose=-1, **p)
    raise KeyError(name)

DEFAULTS_DEEP = {
    "dnn": {"h1": 128, "h2": 64, "d1": .3, "d2": .2, "lr": 1e-3, "bs": 256},
    "cnn": {"f1": 64, "f2": 32, "dense": 64, "drop": .3, "lr": 1e-3, "bs": 256},
    "rnn": {"units": 64, "dense": 64, "drop": .3, "lr": 1e-3, "bs": 256},
    "ftt": {"d": 64, "heads": 8, "blocks": 3, "ff": 128, "drop": .1, "lr": 5e-4, "bs": 256}}

def build_dnn(F, p=None):
    q = {**DEFAULTS_DEEP["dnn"], **(p or {})}
    return models.Sequential([layers.Input((F,)),
        layers.Dense(q["h1"], activation="relu"), layers.Dropout(q["d1"]),
        layers.Dense(q["h2"], activation="relu"), layers.Dropout(q["d2"]),
        layers.Dense(C, activation="softmax")], name="dnn")
def build_cnn(F, p=None):
    q = {**DEFAULTS_DEEP["cnn"], **(p or {})}
    return models.Sequential([layers.Input((F, 1)),
        layers.Conv1D(q["f1"], 3, activation="relu", padding="same"), layers.MaxPooling1D(2),
        layers.Conv1D(q["f2"], 3, activation="relu", padding="same"), layers.Flatten(),
        layers.Dense(q["dense"], activation="relu"), layers.Dropout(q["drop"]),
        layers.Dense(C, activation="softmax")], name="cnn")
def build_rnn(F, p=None):
    q = {**DEFAULTS_DEEP["rnn"], **(p or {})}
    return models.Sequential([layers.Input((F, 1)),
        layers.SimpleRNN(q["units"], activation="relu"), layers.Dropout(q["drop"]),
        layers.Dense(q["dense"], activation="relu"),
        layers.Dense(C, activation="softmax")], name="rnn")

class FeatureTokenizer(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        F = int(shape[-1])
        self.w = self.add_weight(shape=(F, self.d), initializer="glorot_uniform", name="w")
        self.b = self.add_weight(shape=(F, self.d), initializer="zeros", name="b")
    def call(self, x): return x[:, :, None] * self.w + self.b
    def get_config(self): return {**super().get_config(), "d": self.d}
class ClsToken(layers.Layer):
    def __init__(self, d, **kw): super().__init__(**kw); self.d = d
    def build(self, shape):
        self.cls = self.add_weight(shape=(1, 1, self.d), initializer="glorot_uniform", name="cls")
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): return {**super().get_config(), "d": self.d}
def build_ftt(F, p=None):
    q = {**DEFAULTS_DEEP["ftt"], **(p or {})}
    d, heads, blocks, ff, drop = q["d"], q["heads"], q["blocks"], q["ff"], q["drop"]
    inp = layers.Input((F,)); x = ClsToken(d)(FeatureTokenizer(d)(inp))
    for _ in range(blocks):
        h = layers.LayerNormalization()(x)
        h = layers.MultiHeadAttention(num_heads=heads, key_dim=max(1, d // heads), dropout=drop)(h, h)
        x = layers.Add()([x, h])
        h = layers.LayerNormalization()(x)
        h = layers.Dense(ff, activation="gelu")(h); h = layers.Dropout(drop)(h)
        x = layers.Add()([x, layers.Dense(d)(h)])
    return models.Model(inp, layers.Dense(C, activation="softmax")(
        layers.LayerNormalization()(x[:, 0])), name="ftt")

DEEP = {"rnn": build_rnn, "cnn": build_cnn, "dnn": build_dnn, "ftt": build_ftt}
CUSTOM_OBJECTS = {"FeatureTokenizer": FeatureTokenizer, "ClsToken": ClsToken}
def shape_for(name, A): return A if name in ("dnn", "ftt") else A.reshape(-1, A.shape[1], 1)
def build_ae(F):
    return models.Sequential([layers.Input((F,)),
        layers.Dense(64, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"), layers.Dense(F)], name="ae")
ALL_MODELS = FAST_NAMES + DEEP_NAMES
print("modeles definis :", ALL_MODELS, "+ autoencodeur")


In [ ]:
# 4.2 — Moteur : progression, evaluation, persistance, execution par lots
class Progress:
    def __init__(self, total, titre=""):
        self.total, self.i, self.t0 = total, 0, time.time()
        if total: print(f"\n=== {titre} : {total} modele(s) a entrainer ===", flush=True)
    def start(self, name):
        self.i += 1; el = time.time() - self.t0
        eta = (el / max(self.i - 1, 1)) * (self.total - self.i + 1) if self.i > 1 else None
        msg = f"[{self.i:>3}/{self.total}] > {name:38s} RAM {mem():4.1f} Go | ecoule {fmt(el)}"
        if eta: msg += f" | reste ~{fmt(eta)}"
        print(msg, flush=True)
    def stop(self, r, dt):
        fpr = r["binary"]["fpr"]
        print(f"           OK  acc {r['accuracy']:.4f}  mF1 {r['macro_f1']:.4f}  "
              f"MCC {r['mcc']:.4f}  FPR {(f'{fpr:.4%}' if fpr is not None else 'n/a'):>8s}  "
              f"({fmt(dt)})", flush=True)

class EpochLog(callbacks.Callback):
    def __init__(self, total_epochs=30): super().__init__(); self.n = total_epochs
    def on_epoch_begin(self, epoch, logs=None): self._t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f"           epoque {epoch+1:>2}/{self.n}  loss {logs.get('loss', 0):.4f}  "
              f"val_loss {logs.get('val_loss', 0):.4f}  ({time.time()-self._t:.0f} s)", flush=True)

def make_xy(cols, key):
    tr_, va_, te_ = splits[key]; ci = cols_idx(cols)
    sc = RobustScaler().fit(X_ALL[tr_][:, ci])            # scaler ajuste sur le TRAIN seul
    parts = [np.nan_to_num(sc.transform(X_ALL[i_][:, ci]), nan=0., posinf=0., neginf=0.).astype(np.float32)
             for i_ in (tr_, va_, te_)]
    return parts, (y[tr_], y[va_], y[te_]), sc

def evaluate(y_true, probs, fit_t, pred_t):
    pred = probs.argmax(1); present = np.unique(y_true)
    is_att, pred_att = y_true != BENIGN_IDX, pred != BENIGN_IDX
    p_att = 1.0 - probs[:, BENIGN_IDX]
    return {"accuracy": float((pred == y_true).mean()),
            "macro_f1": float(f1_score(y_true, pred, labels=present, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, pred)),
            "per_class_f1": {CLASS_NAMES[i]: float(v) for i, v in zip(range(C),
                f1_score(y_true, pred, labels=range(C), average=None, zero_division=0))},
            "binary": {"detection_f1": float(f1_score(is_att, pred_att, zero_division=0)),
                       "fpr": float(pred_att[~is_att].mean()) if (~is_att).any() else None,
                       "fnr": float((~pred_att[is_att]).mean()) if is_att.any() else None,
                       "pr_auc": float(average_precision_score(is_att, p_att)) if 0 < is_att.mean() < 1 else None,
                       "roc_auc": float(roc_auc_score(is_att, p_att)) if 0 < is_att.mean() < 1 else None},
            "fit_time_s": round(fit_t, 2), "predict_time_s": round(pred_t, 3),
            "n_test": int(len(y_true))}

def load_probs(key):
    z = np.load(kfile(key)); return z["probs_val"].astype(np.float64), z["probs_test"].astype(np.float64)
def record(key, y_true, pva, pte, fit_t, pred_t):
    RESULTS["models"][key] = evaluate(y_true, pte, fit_t, pred_t)
    np.savez_compressed(kfile(key), probs_val=pva.astype(np.float16), probs_test=pte.astype(np.float16))
    save_results(); return RESULTS["models"][key]

REF_SPLIT = "strat_seed1"
def fit_sk(mname, Xtr, ytr, params=None):
    if mname == "knn" and len(ytr) > KNN_MAX_TRAIN:      # budget memoire declare
        sel, _ = train_test_split(np.arange(len(ytr)), train_size=KNN_MAX_TRAIN,
                                  random_state=0, stratify=ytr)
        Xtr, ytr = Xtr[sel], ytr[sel]
    present = np.unique(ytr); model = make_sk(mname, params)
    model.fit(Xtr, np.searchsorted(present, ytr) if len(present) < C else ytr)
    def predict_full(X):
        p = model.predict_proba(X)
        if len(present) == C and list(getattr(model, "classes_", range(C))) == list(range(C)):
            return p
        out = np.zeros((len(X), C), dtype=np.float32)
        out[:, (present if len(present) < C else np.asarray(model.classes_, int))] = p
        return out
    return model, predict_full

def class_weights_safe(ytr):
    present = np.unique(ytr)
    w = compute_class_weight("balanced", classes=present, y=ytr)
    cw = {int(c): 1.0 for c in range(C)}
    cw.update({int(c): float(v) for c, v in zip(present, w)}); return cw

def train_one(m, tag, sk_, arm, XY, params=None, pg=None):
    key = run_key(m, tag, sk_, arm)
    (Xtr, Xva, Xte), (ytr, yva, yte) = XY
    if pg: pg.start(f"{m}{'' if arm=='default' else ' (regle)'} | {tag} | {sk_}")
    t_all = time.time()
    if m in DEEP:
        q = {**DEFAULTS_DEEP[m], **(params or {})}
        tf.keras.utils.set_random_seed(int(sk_[-1]) if sk_[-1].isdigit() else 1)
        mdl = DEEP[m](Xtr.shape[1], q)
        mdl.compile(optimizer=tf.keras.optimizers.Adam(q["lr"]),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        t0 = time.time()
        h = mdl.fit(shape_for(m, Xtr), ytr, validation_data=(shape_for(m, Xva), yva),
                    epochs=30, batch_size=q["bs"], class_weight=class_weights_safe(ytr),
                    verbose=0, callbacks=[EpochLog(30),
                        callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                restore_best_weights=True)])
        fit_t = time.time() - t0
        t0 = time.time(); pte = mdl.predict(shape_for(m, Xte), batch_size=1024, verbose=0)
        pred_t = time.time() - t0
        pva = mdl.predict(shape_for(m, Xva), batch_size=1024, verbose=0)
        RESULTS["history"][key] = {k: [float(x) for x in v] for k, v in h.history.items()}
        if SAVE_MODELS and sk_ == REF_SPLIT and tag in ("audited", "clean"):
            mdl.save(MODELS / f"{m}_{arm}_{tag}_{sk_}.keras")
        del mdl; tf.keras.backend.clear_session()
    else:
        t0 = time.time(); mdl, pf = fit_sk(m, Xtr, ytr, params); fit_t = time.time() - t0
        t0 = time.time(); pte = pf(Xte); pred_t = time.time() - t0
        pva = pf(Xva)
        if SAVE_MODELS and sk_ == REF_SPLIT and tag in ("audited", "clean"):
            joblib.dump(mdl, MODELS / f"{m}_{arm}_{tag}_{sk_}.joblib", compress=3)
        del mdl, pf
    r = record(key, yte, pva, pte, fit_t, pred_t)
    if pg: pg.stop(r, time.time() - t_all)
    del pva, pte; gc.collect()

def run_batch(keep, titre, params_of=None):
    # Entraine les runs planifies qui satisfont `keep` et ne sont pas deja faits.
    # Les matrices normalisees sont calculees UNE fois par (condition, split).
    cols_of = {"full": F_FULL, "clean": F_CLEAN, "audited": None}   # 'audited' resolu au vol
    todo = [r for r in plan_runs() if keep(r) and not done(run_key(*r))]
    if not todo:
        print(f"=== {titre} : rien a faire, tout est deja calcule ==="); return
    todo.sort(key=lambda r: (r[1], r[2], r[3]))
    pg = Progress(len(todo), titre)
    for (tag, sk_), grp in itertools.groupby(todo, key=lambda r: (r[1], r[2])):
        grp = list(grp)
        cols = F_AUDIT if tag == "audited" else cols_of[tag]
        XY = make_xy(cols, sk_)[:2]
        for (m, _, _, arm) in grp:
            p = (params_of or {}).get(m) if arm == "tuned" else None
            train_one(m, tag, sk_, arm, XY, p, pg)
        del XY; gc.collect()
    print(f"=== {titre} : termine ({fmt(time.time()-pg.t0)}) — RAM {mem():.1f} Go ===")
print("moteur pret")


## 5. Entraînement, par lots indépendants

Chaque cellule ci-dessous est **autonome** : elle ne calcule que ce qui manque et peut être
relancée seule après une déconnexion. Les lots « rapides » tournent confortablement sur
CPU ; les lots « profonds » réclament le GPU.


In [ ]:
# 5.1a — AVANT AUDIT, modeles rapides (CPU suffisant)
run_batch(lambda r: r[1] in ("full", "clean") and r[0] in FAST_NAMES, "Avant audit — rapides")


In [ ]:
# 5.1b — AVANT AUDIT, modeles profonds (GPU recommande)
run_batch(lambda r: r[1] in ("full", "clean") and r[0] in DEEP_NAMES, "Avant audit — profonds")


In [ ]:
# 5.2 — AUDIT : transferabilite mono-feature sur TOUTES les features -> LISTE NOIRE
from sklearn.inspection import permutation_importance

# Deux mesures complementaires, et l'ordre compte :
#  (a) importance par permutation = ce que la litterature utilise. AVEUGLE aux raccourcis
#      REDONDANTS : si k features portent le meme raccourci, en permuter une seule ne coute
#      rien et leur importance est nulle.
#  (b) transferabilite mono-feature = notre test. Chaque feature est evaluee SEULE sous les
#      deux protocoles ; le rapport acc(temporel)/acc(stratifie) mesure si son pouvoir
#      predictif tient hors du calendrier de capture. Applique a TOUTES les features.
AUDIT_VERSION   = "v3-full-scan"
SHORTCUT_RATIO  = 0.50    # seuil : ratio de transferabilite en dessous duquel on ecarte
SHORTCUT_MIN_ACC = 3.0    # ... a condition que la feature soit predictive seule (x hasard)

if RESULTS.get("audit", {}).get("version") != AUDIT_VERSION:
    RESULTS.pop("audit", None)
    chance = RESULTS["shortcut_probes"]["chance_majority"]

    # (a) importance par permutation, pour la figure d'annexe et la demonstration de sa limite
    print("importance par permutation (reference LightGBM)…", flush=True)
    (Xtr, Xva, Xte), (ytr, yva, yte), _ = make_xy(F_CLEAN, "strat_seed1")
    ref = LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=.1,
                         n_jobs=2, random_state=0, verbose=-1).fit(Xtr, ytr)
    rng = np.random.RandomState(0)
    six = rng.choice(len(yte), size=min(15000, len(yte)), replace=False)
    imp = permutation_importance(ref, Xte[six], yte[six], n_repeats=5, random_state=0,
                                 n_jobs=1, scoring="accuracy")
    PERM = {F_CLEAN[i]: (float(imp.importances_mean[i]), float(imp.importances_std[i]))
            for i in range(len(F_CLEAN))}
    del ref, Xtr, Xva, Xte; gc.collect()

    # (b) balayage COMPLET : chaque feature de la condition clean, sous les deux protocoles
    print(f"balayage de transferabilite sur les {len(F_CLEAN)} features…", flush=True)
    rows, t_scan = [], time.time()
    for k, feat in enumerate(F_CLEAN, 1):
        a_s, a_t = probe([feat], "strat_seed1"), probe([feat], "temporal")
        ratio = a_t / a_s if a_s > 0 else 1.0
        short = (a_s > SHORTCUT_MIN_ACC * chance) and (ratio < SHORTCUT_RATIO)
        rows.append({"feature": feat, "importance": round(PERM[feat][0], 4),
                     "acc seule (stratifie)": round(a_s, 3),
                     "acc seule (temporel)": round(a_t, 3),
                     "transferabilite": round(ratio, 2),
                     "raccourci": "OUI" if short else "-"})
        if k % 10 == 0 or short:
            print(f"  [{k:>2}/{len(F_CLEAN)}] {feat:22s} {a_s:.3f} -> {a_t:.3f}  "
                  f"ratio {ratio:.2f}{'   <-- RACCOURCI' if short else ''}", flush=True)
    flagged = [r["feature"] for r in rows if r["raccourci"] == "OUI"]

    # (c) colonnes redondantes parmi les features ecartees : un raccourci duplique est
    #     invisible a l'importance par permutation — on le documente explicitement.
    dup = []
    if flagged:
        fi = cols_idx(flagged); M = X_ALL[:, fi]
        for a in range(len(flagged)):
            for b in range(a + 1, len(flagged)):
                if np.allclose(M[:, a], M[:, b], rtol=1e-5, atol=1e-8):
                    dup.append([flagged[a], flagged[b]])
        del M; gc.collect()

    RESULTS["audit"] = {"version": AUDIT_VERSION, "chance": chance,
                        "rule": {"ratio_below": SHORTCUT_RATIO, "min_acc_x_chance": SHORTCUT_MIN_ACC},
                        "perm_importance": {k: v[0] for k, v in PERM.items()},
                        "transfer_table": rows, "duplicate_pairs": dup,
                        "blacklist": POSITIONAL + flagged,
                        "scan_time_s": round(time.time() - t_scan, 1)}
    save_results()
    print(f"balayage termine en {fmt(time.time()-t_scan)}")

A = RESULTS["audit"]
tab = pd.DataFrame(A["transfer_table"]).sort_values("transferabilite")
display(tab.head(20))
BLACKLIST = A["blacklist"]
F_AUDIT   = [c for c in F_CLEAN if c not in BLACKLIST]
RESULTS["slice60"]["features_audited"] = F_AUDIT; save_results()
print(f"\nLISTE NOIRE ({len(BLACKLIST)}) : {BLACKLIST}")
print(f"features auditees conservees : {len(F_AUDIT)} / {len(F_FULL)}")
if A["duplicate_pairs"]:
    print(f"colonnes DUPLIQUEES parmi les features ecartees : {A['duplicate_pairs']}")
n_zero = sum(1 for r in A["transfer_table"]
             if r["raccourci"] == "OUI" and abs(r["importance"]) < 1e-4)
print(f"raccourcis d'importance par permutation quasi nulle : {n_zero}/{len(BLACKLIST)-len(POSITIONAL)}"
      "  <-- invisibles a la methode usuelle")

# Figure 8 : plan de transferabilite
fig, ax = plt.subplots(figsize=(5.8, 4.8))
for r_ in A["transfer_table"]:
    sh = r_["raccourci"] == "OUI"
    ax.scatter(r_["acc seule (stratifie)"], r_["acc seule (temporel)"],
               s=38 if sh else 22, color="#C44E52" if sh else "#4C72B0", zorder=3 if sh else 2)
    if sh or r_["acc seule (stratifie)"] > .75:
        ax.annotate(r_["feature"], (r_["acc seule (stratifie)"], r_["acc seule (temporel)"]),
                    fontsize=6, xytext=(3, 3), textcoords="offset points")
xs = np.linspace(0, 1, 50)
ax.plot(xs, xs, "k--", lw=.7, label="transfert parfait")
ax.plot(xs, SHORTCUT_RATIO * xs, color="#C44E52", ls="-.", lw=.8,
        label=f"seuil (ratio {SHORTCUT_RATIO})")
ax.axhline(A["chance"], color="gray", ls=":", lw=.8, label="hasard")
ax.axvline(SHORTCUT_MIN_ACC * A["chance"], color="gray", ls=":", lw=.8)
ax.set_xlabel("accuracy de la feature seule — split stratifie")
ax.set_ylabel("accuracy de la feature seule — split temporel")
ax.set_title("Figure 8 — transferabilite des features (rouge = raccourci)")
ax.legend(fontsize=7, loc="upper left")
plt.savefig(FIGS / "fig8_transferability.png"); plt.savefig(FIGS / "fig8_transferability.pdf"); plt.show()

# Figure 9 : spectre trie des ratios (montre ou se situe le seuil)
srt = sorted(A["transfer_table"], key=lambda r: r["transferabilite"])
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(len(srt)), [r["transferabilite"] for r in srt],
       color=["#C44E52" if r["raccourci"] == "OUI" else "#4C72B0" for r in srt])
ax.axhline(SHORTCUT_RATIO, color="k", ls="--", lw=.8, label=f"seuil {SHORTCUT_RATIO}")
ax.set_xticks(range(len(srt)))
ax.set_xticklabels([r["feature"] for r in srt], rotation=90, fontsize=5)
ax.set_ylabel("transferabilite\n(acc temporel / acc stratifie)")
ax.set_title("Figure 9 — spectre de transferabilite des features")
ax.legend(fontsize=8)
plt.savefig(FIGS / "fig9_transferability_spectrum.png")
plt.savefig(FIGS / "fig9_transferability_spectrum.pdf"); plt.show()

# Annexe : importance par permutation (top 20) — et sa cecite aux raccourcis redondants
top = sorted(A["perm_importance"].items(), key=lambda kv: -kv[1])[:20]
fig, ax = plt.subplots(figsize=(6, 5))
nm = [t[0] for t in top][::-1]; vl = [t[1] for t in top][::-1]
ax.barh(nm, vl, color=["#C44E52" if n_ in BLACKLIST else "#4C72B0" for n_ in nm])
ax.set_xlabel("chute d'accuracy par permutation")
ax.set_title("A6 — importance par permutation (rouge = ecarte par l'audit)")
plt.savefig(SUPP / "A6_permutation_importance.png"); plt.savefig(SUPP / "A6_permutation_importance.pdf"); plt.show()
gc.collect()


In [ ]:
# 5.3a — APRES AUDIT, modeles rapides (CPU suffisant) — tableau principal
run_batch(lambda r: r[1] == "audited" and r[3] == "default" and r[0] in FAST_NAMES,
          "Apres audit — rapides")


In [ ]:
# 5.3b — APRES AUDIT, modeles profonds (GPU recommande) — tableau principal
run_batch(lambda r: r[1] == "audited" and r[3] == "default" and r[0] in DEEP_NAMES,
          "Apres audit — profonds")


### 5.4 Recherche d'hyperparamètres à budget déclaré

Recherche aléatoire, sélection sur la **validation** (jamais le test), condition `audited`,
graine 1. Budget : $N$ tirages **et** un plafond de temps identique pour tous — le premier
atteint arrête la recherche. Le **tirage 0 est la configuration par défaut**, si bien que le
réglage ne peut jamais dégrader un modèle. La configuration gagnante est ensuite réentraînée
sur le train complet (§5.5).


In [ ]:
# 5.4 — Recherche (peut etre relancee : chaque modele est memorise des qu'il est fini)
HPO_TRIALS_FAST, HPO_TRIALS_DEEP = 20, 10
HPO_TIME_CAP_S, HPO_SUB_TRAIN, HPO_SUB_VAL = 900, 60000, 30000

SPACE_SK = {
    "logreg":   lambda r: {"C": float(10 ** r.uniform(-3, 2)), "max_iter": 1000},
    "nb":       lambda r: {"var_smoothing": float(10 ** r.uniform(-11, -6))},
    "knn":      lambda r: {"n_neighbors": int(r.choice([3, 5, 7, 11, 15, 21])),
                           "weights": str(r.choice(["uniform", "distance"])),
                           "p": int(r.choice([1, 2]))},
    "rf":       lambda r: {"n_estimators": int(r.choice([100, 200, 300])),
                           "max_depth": (None if r.rand() < .3 else int(r.choice([8, 16, 24, 32]))),
                           "min_samples_leaf": int(r.choice([1, 2, 5, 10])),
                           "max_features": str(r.choice(["sqrt", "log2"]))},
    "xgboost":  lambda r: {"n_estimators": int(r.choice([200, 300, 500])),
                           "max_depth": int(r.choice([4, 6, 8, 10, 12])),
                           "learning_rate": float(10 ** r.uniform(-2, -.7)),
                           "subsample": float(r.choice([.6, .8, 1.])),
                           "colsample_bytree": float(r.choice([.6, .8, 1.])),
                           "min_child_weight": int(r.choice([1, 5, 20]))},
    "lightgbm": lambda r: {"n_estimators": int(r.choice([200, 300, 500])),
                           "num_leaves": int(r.choice([31, 63, 127])),
                           "learning_rate": float(10 ** r.uniform(-2, -.7)),
                           "min_child_samples": int(r.choice([5, 20, 50, 100])),
                           "subsample": float(r.choice([.6, .8, 1.])),
                           "colsample_bytree": float(r.choice([.6, .8, 1.]))}}
SPACE_DEEP = {
    "dnn": lambda r: {"h1": int(r.choice([64, 128, 256])), "h2": int(r.choice([32, 64, 128])),
                      "d1": float(r.choice([.1, .2, .3, .5])), "d2": float(r.choice([0., .1, .2, .3])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "cnn": lambda r: {"f1": int(r.choice([32, 64, 128])), "f2": int(r.choice([16, 32, 64])),
                      "dense": int(r.choice([32, 64, 128])), "drop": float(r.choice([.1, .2, .3, .5])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "rnn": lambda r: {"units": int(r.choice([32, 64, 128])), "dense": int(r.choice([32, 64, 128])),
                      "drop": float(r.choice([.1, .2, .3, .5])),
                      "lr": float(10 ** r.uniform(-4, -2.3)), "bs": int(r.choice([128, 256, 512]))},
    "ftt": lambda r: {"d": int(r.choice([32, 64])), "heads": int(r.choice([4, 8])),
                      "blocks": int(r.choice([2, 3])), "ff": int(r.choice([64, 128])),
                      "drop": float(r.choice([0., .1, .2])),
                      "lr": float(10 ** r.uniform(-4, -2.7)), "bs": int(r.choice([256, 512]))}}

def _sub(X, yy, n):
    if len(yy) <= n: return X, yy
    ix, _ = train_test_split(np.arange(len(yy)), train_size=n, random_state=0, stratify=yy)
    return X[ix], yy[ix]
def macro_f1(y_true, pred):
    return float(f1_score(y_true, pred, labels=np.unique(y_true), average="macro", zero_division=0))

def hpo_model(mname, XY):
    (Xtr, Xva, _), (ytr, yva, _) = XY
    Xs, ys = _sub(Xtr, ytr, HPO_SUB_TRAIN); Xv, yv = _sub(Xva, yva, HPO_SUB_VAL)
    is_deep = mname in DEEP
    space  = (SPACE_DEEP if is_deep else SPACE_SK)[mname]
    budget = HPO_TRIALS_DEEP if is_deep else HPO_TRIALS_FAST
    r = np.random.RandomState(0); t0 = time.time(); trials = []
    cands = [{}] + [space(r) for _ in range(budget - 1)]      # tirage 0 = defaut
    for i, p in enumerate(cands):
        if time.time() - t0 > HPO_TIME_CAP_S:
            print(f"      plafond atteint apres {i} tirages", flush=True); break
        try:
            if is_deep:
                q = {**DEFAULTS_DEEP[mname], **p}
                tf.keras.utils.set_random_seed(1)
                mdl = DEEP[mname](Xs.shape[1], q)
                mdl.compile(optimizer=tf.keras.optimizers.Adam(q["lr"]),
                            loss="sparse_categorical_crossentropy")
                mdl.fit(shape_for(mname, Xs), ys, validation_data=(shape_for(mname, Xv), yv),
                        epochs=12, batch_size=q["bs"], class_weight=class_weights_safe(ys),
                        verbose=0, callbacks=[callbacks.EarlyStopping(monitor="val_loss",
                                              patience=3, restore_best_weights=True)])
                pred = mdl.predict(shape_for(mname, Xv), batch_size=1024, verbose=0).argmax(1)
                del mdl; tf.keras.backend.clear_session()
            else:
                mdl, pf = fit_sk(mname, Xs, ys, p); pred = pf(Xv).argmax(1); del mdl, pf
            sc = macro_f1(yv, pred); trials.append({"params": p, "val_macro_f1": sc})
            print(f"      tirage {i+1:>2}/{budget}  macro-F1 {sc:.4f}"
                  f"{'  (defaut)' if i == 0 else ''}", flush=True)
        except Exception as e:
            trials.append({"params": p, "val_macro_f1": None, "error": str(e)[:150]})
            print(f"      tirage {i+1:>2}/{budget}  echec : {str(e)[:70]}", flush=True)
        gc.collect()
    ok = [t for t in trials if t["val_macro_f1"] is not None]
    best = max(ok, key=lambda t: t["val_macro_f1"])
    return {"best_params": best["params"], "best_val_macro_f1": best["val_macro_f1"],
            "default_val_macro_f1": ok[0]["val_macro_f1"], "n_trials": len(trials),
            "search_time_s": round(time.time() - t0, 1), "trials": trials}

if RUN_HPO:
    RESULTS.setdefault("hpo", {})
    todo = [m for m in TUNABLE if m not in RESULTS["hpo"]]
    pg = Progress(len(todo), "Recherche d'hyperparametres")
    XY_HPO = make_xy(F_AUDIT, "strat_seed1")[:2]
    for m in todo:
        pg.start(f"recherche {m}")
        RESULTS["hpo"][m] = h = hpo_model(m, XY_HPO)
        print(f"           defaut {h['default_val_macro_f1']:.4f} -> regle {h['best_val_macro_f1']:.4f}"
              f"  ({h['n_trials']} tirages, {fmt(h['search_time_s'])})")
        print(f"           {h['best_params']}", flush=True)
        save_results(); gc.collect()
    del XY_HPO; gc.collect()

if "hpo" in RESULTS:
    display(pd.DataFrame([{"modele": m, "defaut": round(h["default_val_macro_f1"], 4),
                           "regle": round(h["best_val_macro_f1"], 4),
                           "gain": round(h["best_val_macro_f1"] - h["default_val_macro_f1"], 4),
                           "tirages": h["n_trials"], "temps (s)": h["search_time_s"]}
                          for m, h in RESULTS["hpo"].items()]).set_index("modele"))
    (MODELS / "hpo_best_params.json").write_text(json.dumps(
        {m: h["best_params"] for m, h in RESULTS["hpo"].items()}, indent=1), encoding="utf-8")


In [ ]:
# 5.5a — BRAS REGLE, modeles rapides (CPU suffisant)
TUNED = {m: h["best_params"] for m, h in RESULTS.get("hpo", {}).items()}
run_batch(lambda r: r[3] == "tuned" and r[0] in FAST_NAMES, "Bras regle — rapides", TUNED)


In [ ]:
# 5.5b — BRAS REGLE, modeles profonds (GPU recommande)
TUNED = {m: h["best_params"] for m, h in RESULTS.get("hpo", {}).items()}
run_batch(lambda r: r[3] == "tuned" and r[0] in DEEP_NAMES, "Bras regle — profonds", TUNED)


In [ ]:
# 5.6 — Autoencodeur : entraine sur le BENIN seul, evalue en AUROC
if "autoencoder" not in RESULTS:
    ae_res = {}
    pg = Progress(len(SEEDS), "Autoencodeur")
    for s in SEEDS:
        pg.start(f"autoencodeur | graine {s}")
        (Xtr, Xva, Xte), (ytr, yva, yte), _ = make_xy(F_AUDIT, f"strat_seed{s}")
        tf.keras.utils.set_random_seed(s)
        ae = build_ae(Xtr.shape[1]); ae.compile(optimizer="adam", loss="mse")
        bt, bv = Xtr[ytr == BENIGN_IDX], Xva[yva == BENIGN_IDX]
        ae.fit(bt, bt, validation_data=(bv, bv), epochs=50, batch_size=256, verbose=0,
               callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                  restore_best_weights=True)])
        err = np.mean((ae.predict(Xte, batch_size=1024, verbose=0) - Xte) ** 2, axis=1)
        per_fam = {CLASS_NAMES[i]: float(roc_auc_score(
                       yte[(yte == i) | (yte == BENIGN_IDX)] == i,
                       err[(yte == i) | (yte == BENIGN_IDX)]))
                   for i in range(C) if i != BENIGN_IDX and (yte == i).any()}
        g_ = float(roc_auc_score(yte != BENIGN_IDX, err))
        ae_res[f"seed{s}"] = {"auroc_global": g_, "auroc_per_family": per_fam}
        if s == 1:
            if SAVE_MODELS: ae.save(MODELS / "ae_default_audited_strat_seed1.keras")
            np.savez_compressed(SAVE / "ae_scores_seed1.npz", err=err.astype(np.float32), y=yte)
        print(f"           AUROC {g_:.4f}", flush=True)
        del ae, Xtr, Xva, Xte, err; tf.keras.backend.clear_session(); gc.collect()
    RESULTS["autoencoder"] = ae_res; save_results()

g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
print(f"AUROC global : {np.mean(g):.4f} +/- {np.std(g):.4f}")
fam = pd.DataFrame({k: v["auroc_per_family"] for k, v in RESULTS["autoencoder"].items()})
display(fam.assign(moyenne=fam.mean(axis=1).round(4)).round(4))


## 6. Évaluation, figures et interprétation

Figures de l'article dans `figures/` (Fig. 1–8), figures d'annexe pour le rapport de thèse
dans `figures_annexe/` (A1–A15), fragments LaTeX dans `tables/`, en PNG et PDF à 300 dpi.


In [ ]:
# 6.1 — Tableau principal et tables LaTeX
def agg(m, tag, metric="macro_f1"):
    v = [RESULTS["models"][f"{m}|{tag}|strat_seed{s}"][metric]
         for s in SEEDS if f"{m}|{tag}|strat_seed{s}" in RESULTS["models"]]
    return (float(np.mean(v)), float(np.std(v))) if v else (np.nan, np.nan)
def one(m, tag, split, metric="macro_f1"):
    return RESULTS["models"].get(f"{m}|{tag}|{split}", {}).get(metric, np.nan)
def pm(mu, sd): return "--" if np.isnan(mu) else f"{mu:.4f} +/- {sd:.4f}"

rows = []
for m in ALL_MODELS:
    fpr = RESULTS["models"].get(f"{m}|audited|strat_seed1", {}).get("binary", {}).get("fpr")
    rows.append({"modele": m,
                 "full (s1)": round(one(m, "full", "strat_seed1"), 4),
                 "clean (s1)": round(one(m, "clean", "strat_seed1"), 4),
                 "audited defaut (5g)": pm(*agg(m, "audited")),
                 "audited regle (5g)": pm(*agg(m + "#tuned", "audited")),
                 "audited defaut (temporel)": round(one(m, "audited", "temporal"), 4),
                 "audited regle (temporel)": round(one(m + "#tuned", "audited", "temporal"), 4),
                 "MCC (s1)": round(one(m, "audited", "strat_seed1", "mcc"), 4),
                 "FPR (s1)": (round(fpr, 5) if fpr is not None else None)})
main_tab = pd.DataFrame(rows).set_index("modele"); display(main_tab)
RESULTS["main_table"] = main_tab.reset_index().to_dict("records"); save_results()

def esc(x): return "--" if (isinstance(x, float) and np.isnan(x)) else (
    f"{x:.4f}" if isinstance(x, float) else str(x).replace("+/-", "$\\pm$"))
(TABS / "t1_intervals.tex").write_text("\n".join(
    f"{iv}\\,s & {RESULTS['interval_stats'][iv]['total']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign']:,} & "
    f"{RESULTS['interval_stats'][iv]['benign_share']*100:.1f}\\% \\\\" for iv in INTERVALS))
(TABS / "t2_main.tex").write_text("\n".join(
    " & ".join([i] + [esc(r[c]) for c in main_tab.columns]) + " \\\\" for i, r in main_tab.iterrows()))
if "hpo" in RESULTS:
    (TABS / "t3_hpo.tex").write_text("\n".join(
        f"{m} & {h['n_trials']} & {h['search_time_s']:.0f} & "
        f"{h['default_val_macro_f1']:.4f} & {h['best_val_macro_f1']:.4f} \\\\"
        for m, h in RESULTS["hpo"].items()))
print("tables :", sorted(p.name for p in TABS.glob("*.tex")))


In [ ]:
# 6.2 — Figure 2 : effet de l'audit et du protocole temporel
CONDS = [("full\n(stratifie)",    lambda m: one(m, "full", "strat_seed1")),
         ("clean\n(stratifie)",   lambda m: one(m, "clean", "strat_seed1")),
         ("audited\n(stratifie)", lambda m: agg(m, "audited")[0]),
         ("clean\n(temporel)",    lambda m: one(m, "clean", "temporal")),
         ("audited\n(temporel)",  lambda m: one(m, "audited", "temporal"))]
fig, ax = plt.subplots(figsize=(8, 4.6))
for m in ALL_MODELS:
    if m == "majority": continue
    ax.plot(range(len(CONDS)), [f(m) for _, f in CONDS], marker="o", ms=4, lw=1.2, label=m)
ax.set_xticks(range(len(CONDS))); ax.set_xticklabels([c for c, _ in CONDS])
ax.set_ylabel("macro-F1"); ax.set_ylim(bottom=0)
ax.set_title("Figure 2 — effet de l'audit et du protocole temporel sur le classement")
ax.legend(fontsize=7, ncol=2, loc="lower left")
plt.savefig(FIGS / "fig2_before_after.png"); plt.savefig(FIGS / "fig2_before_after.pdf"); plt.show()


In [ ]:
# 6.3 — Calibration : temperature scaling, ECE, diagrammes de fiabilite (Figure 4)
from scipy.optimize import minimize_scalar
def fit_temperature(pva, yva):
    logp = np.log(np.clip(pva, 1e-12, 1))
    def nll(T):
        q = logp / T; q -= q.max(1, keepdims=True)
        p = np.exp(q); p /= p.sum(1, keepdims=True)
        return -np.mean(np.log(np.clip(p[np.arange(len(yva)), yva], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(.05, 10.), method="bounded").x)
def apply_T(p, T):
    q = np.log(np.clip(p, 1e-12, 1)) / T; q -= q.max(1, keepdims=True)
    e = np.exp(q); return e / e.sum(1, keepdims=True)
def ece(p, y_true, bins=ECE_BINS):
    conf, acc = p.max(1), (p.argmax(1) == y_true).astype(float)
    ed = np.linspace(0, 1, bins + 1); out = 0.
    for lo, hi in zip(ed[:-1], ed[1:]):
        m_ = (conf > lo) & (conf <= hi)
        if m_.any(): out += m_.mean() * abs(acc[m_].mean() - conf[m_].mean())
    return float(out)

MAIN = [m for m in ALL_MODELS if m != "majority"]
yva_ref, yte_ref = y[splits["strat_seed1"][1]], y[splits["strat_seed1"][2]]
if "calibration" not in RESULTS:
    cal = {}
    for m in MAIN:
        key = f"{m}|audited|strat_seed1"
        if not done(key): continue
        pva, pte = load_probs(key); T = fit_temperature(pva, yva_ref)
        cal[m] = {"T": T, "ece_before": ece(pte, yte_ref), "ece_after": ece(apply_T(pte, T), yte_ref)}
    RESULTS["calibration"] = cal; save_results()
display(pd.DataFrame(RESULTS["calibration"]).T.round(4))

ncol = 5; nrow = int(np.ceil(len(MAIN) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.7 * nrow), sharex=True, sharey=True)
for ax, m in zip(np.atleast_1d(axes).flat, MAIN):
    key = f"{m}|audited|strat_seed1"
    if not done(key) or m not in RESULTS["calibration"]: ax.axis("off"); continue
    _, pte = load_probs(key); T = RESULTS["calibration"][m]["T"]
    for p, lab in [(pte, "brut"), (apply_T(pte, T), "calibre")]:
        conf, pred = p.max(1), p.argmax(1); xs, ys_ = [], []
        for lo, hi in zip(np.linspace(0, 1, 16)[:-1], np.linspace(0, 1, 16)[1:]):
            m2 = (conf > lo) & (conf <= hi)
            if m2.any(): xs.append(conf[m2].mean()); ys_.append((pred[m2] == yte_ref[m2]).mean())
        ax.plot(xs, ys_, marker="o", ms=2.5, label=lab)
    ax.plot([0, 1], [0, 1], "k--", lw=.6); ax.set_title(f"{m} (T={T:.2f})", fontsize=8)
for ax in np.atleast_1d(axes).flat[len(MAIN):]: ax.axis("off")
np.atleast_1d(axes).flat[0].legend(fontsize=7)
fig.suptitle("Figure 4 — diagrammes de fiabilite (condition auditee, graine 1)")
plt.savefig(FIGS / "fig4_reliability.png"); plt.savefig(FIGS / "fig4_reliability.pdf"); plt.show()


In [ ]:
# 6.4 — Multi-intervalles 5/10/30 s + Figure 3  (memoire liberee apres chaque intervalle)
if RUN_INTERVALS:
    RESULTS.setdefault("intervals", {})
    for iv in ["5", "10", "30"]:
        if iv in RESULTS["intervals"]: continue
        print(f"\n=== intervalle {iv} s ===", flush=True)
        d_iv, y9_iv, t_iv = load_slice_df(iv)
        num_iv, _, cleanc, _, _, _ = feature_sets(d_iv)
        audc = [c for c in cleanc if c not in BLACKLIST]
        Xiv = np.ascontiguousarray(num_iv[audc].values, dtype=np.float32)
        stp = pd.to_numeric(d_iv["StartTime"], errors="coerce").values.reshape(-1, 1)
        y_iv = np.array([CLASS_NAMES.index(v) for v in y9_iv])
        del d_iv, num_iv; gc.collect()
        print(f"  {len(y_iv):,} flux | RAM {mem():.1f} Go", flush=True)
        res_iv = {"n": int(len(y_iv)), "benign_share": float((y_iv == BENIGN_IDX).mean()), "runs": {}}
        idx = np.arange(len(y_iv))
        for s in [1, 2, 3]:
            itr, itmp = train_test_split(idx, test_size=.4, random_state=s, stratify=y_iv)
            iva_, ite_ = train_test_split(itmp, test_size=.5, random_state=s, stratify=y_iv[itmp])
            if s == 1:
                dtc = DecisionTreeClassifier(random_state=1).fit(stp[itr], y_iv[itr])
                res_iv["probe_starttime"] = float((dtc.predict(stp[ite_]) == y_iv[ite_]).mean())
                del dtc
            sc = RobustScaler().fit(Xiv[itr])
            Xtr, Xva_, Xte = [np.nan_to_num(sc.transform(Xiv[i_]), nan=0., posinf=0., neginf=0.).astype(np.float32)
                              for i_ in (itr, iva_, ite_)]
            ytr_, yte_ = y_iv[itr], y_iv[ite_]
            for mname in ["lightgbm", "xgboost"]:
                print(f"  [graine {s}] {mname} …", flush=True)
                t0 = time.time(); mdl, pf = fit_sk(mname, Xtr, ytr_); ft = time.time() - t0
                t0 = time.time(); pte = pf(Xte); pt = time.time() - t0
                res_iv["runs"][f"{mname}|seed{s}"] = evaluate(yte_, pte, ft, pt)
                del mdl, pf, pte; gc.collect()
            print(f"  [graine {s}] dnn …", flush=True)
            tf.keras.utils.set_random_seed(s)
            mdl = build_dnn(Xtr.shape[1])
            mdl.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
            t0 = time.time()
            mdl.fit(Xtr, ytr_, validation_data=(Xva_, y_iv[iva_]), epochs=30, batch_size=256,
                    class_weight=class_weights_safe(ytr_), verbose=0,
                    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                       restore_best_weights=True)])
            ft = time.time() - t0
            t0 = time.time(); pte = mdl.predict(Xte, batch_size=1024, verbose=0); pt = time.time() - t0
            res_iv["runs"][f"dnn|seed{s}"] = evaluate(yte_, pte, ft, pt)
            del mdl, pte, Xtr, Xva_, Xte; tf.keras.backend.clear_session(); gc.collect()
            RESULTS["intervals"][iv] = res_iv; save_results()
        del Xiv, stp, y_iv; gc.collect()
        print(f"  intervalle {iv} s termine | RAM {mem():.1f} Go", flush=True)

if RESULTS.get("intervals"):
    det = pd.DataFrame(index=CLASS_NAMES, columns=INTERVALS, dtype=float)
    for iv in INTERVALS:
        if iv == "60":
            pcf = RESULTS["models"].get("lightgbm|audited|strat_seed1", {}).get("per_class_f1", {})
        else:
            pcs = [r["per_class_f1"] for k, r in RESULTS["intervals"].get(iv, {}).get("runs", {}).items()
                   if k.startswith("lightgbm")]
            pcf = {cn: float(np.mean([p[cn] for p in pcs])) for cn in CLASS_NAMES} if pcs else {}
        for cn in CLASS_NAMES: det.loc[cn, iv] = pcf.get(cn, np.nan)
    fig, ax = plt.subplots(figsize=(5.6, 3.9))
    im = ax.imshow(det.values.astype(float), aspect="auto", vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(4)); ax.set_xticklabels([f"{iv} s" for iv in INTERVALS])
    ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES)
    for i in range(C):
        for j in range(4):
            v = det.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v < .6 else "black")
    plt.colorbar(im, label="F1 par classe (LightGBM, audite)")
    ax.set_title("Figure 3 — detectabilite par famille selon l'intervalle")
    plt.savefig(FIGS / "fig3_interval_heatmap.png"); plt.savefig(FIGS / "fig3_interval_heatmap.pdf"); plt.show()


In [ ]:
# 6.5 — Banc de cout CPU (Figure 5)
if RUN_COST and "cost" not in RESULTS:
    cost = {}
    (Xtr, _, Xte), (ytr, _, _), _ = make_xy(F_AUDIT, "strat_seed1")
    n1, nb = 200, 20; x1, xb = Xte[:n1], Xte[:512]
    for mname in FAST_NAMES:
        print(f"  cout {mname} …", flush=True)
        mdl, pf = fit_sk(mname, Xtr, ytr)
        lat = []
        for i in range(n1):
            t0 = time.perf_counter(); pf(x1[i:i+1]); lat.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        for _ in range(nb): pf(xb)
        thr = nb * 512 / (time.perf_counter() - t0)
        buf = io.BytesIO(); pickle.dump(mdl, buf)
        cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                       "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                       "throughput_512": float(thr), "size_mb": buf.getbuffer().nbytes / 1e6}
        del mdl, pf, buf; gc.collect()
    with tf.device("/CPU:0"):
        for mname in DEEP_NAMES:
            path = MODELS / f"{mname}_default_audited_strat_seed1.keras"
            if not path.exists(): continue
            print(f"  cout {mname} …", flush=True)
            mdl = tf.keras.models.load_model(path, compile=False, custom_objects=CUSTOM_OBJECTS)
            xin = (lambda a: a) if mname in ("dnn", "ftt") else (lambda a: a.reshape(-1, a.shape[1], 1))
            mdl.predict(xin(x1[:8]), verbose=0)
            lat = []
            for i in range(n1):
                t0 = time.perf_counter(); mdl.predict(xin(x1[i:i+1]), verbose=0)
                lat.append(time.perf_counter() - t0)
            t0 = time.perf_counter()
            for _ in range(nb): mdl.predict(xin(xb), verbose=0)
            cost[mname] = {"lat_p50_ms": float(np.percentile(lat, 50) * 1e3),
                           "lat_p99_ms": float(np.percentile(lat, 99) * 1e3),
                           "throughput_512": float(nb * 512 / (time.perf_counter() - t0)),
                           "size_mb": path.stat().st_size / 1e6, "params": int(mdl.count_params())}
            del mdl; tf.keras.backend.clear_session(); gc.collect()
    RESULTS["cost"] = cost; save_results()
    del Xtr, Xte; gc.collect()

if "cost" in RESULTS:
    display(pd.DataFrame(RESULTS["cost"]).T.round(3))
    (TABS / "t4_cost.tex").write_text("\n".join(
        f"{m} & {c['lat_p50_ms']:.2f} & {c['lat_p99_ms']:.2f} & "
        f"{c['throughput_512']:,.0f} & {c['size_mb']:.2f} \\\\" for m, c in RESULTS["cost"].items()))
    fig, ax = plt.subplots(figsize=(5.8, 4.2))
    for m, c in RESULTS["cost"].items():
        mf1 = agg(m, "audited")[0]
        if np.isnan(mf1): continue
        ax.scatter(c["throughput_512"], mf1, s=32, color="#4C72B0")
        ax.annotate(m, (c["throughput_512"], mf1), fontsize=7, xytext=(4, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("debit CPU (flux/s, batch 512, log)")
    ax.set_ylabel("macro-F1 (audite, 5 graines)")
    ax.set_title("Figure 5 — compromis cout d'inference / performance")
    plt.savefig(FIGS / "fig5_cost.png"); plt.savefig(FIGS / "fig5_cost.pdf"); plt.show()


In [ ]:
# 6.6 — Statistiques (Figure 7) et matrice de confusion du meilleur modele (Figure 6)
from scipy.stats import chi2 as chi2dist
def mcnemar_p(pa, pb, yt):
    ca, cb = pa == yt, pb == yt
    b_, c_ = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return 1.0 if b_ + c_ == 0 else float(chi2dist.sf((abs(b_ - c_) - 1) ** 2 / (b_ + c_), 1))

if "stats" not in RESULTS:
    preds = {m: load_probs(f"{m}|audited|strat_seed1")[1].argmax(1)
             for m in MAIN if done(f"{m}|audited|strat_seed1")}
    raw = {f"{a}|{b}": mcnemar_p(preds[a], preds[b], yte_ref)
           for a, b in itertools.combinations(sorted(preds), 2)}
    order = sorted(raw, key=raw.get); mt = len(order)
    holm = {k: min(1., raw[k] * (mt - i)) for i, k in enumerate(order)}
    rng = np.random.RandomState(0); boot = {}
    for m, pr_ in preds.items():
        vals = [f1_score(yte_ref[ix], pr_[ix], average="macro", zero_division=0)
                for ix in (rng.randint(0, len(yte_ref), len(yte_ref)) for _ in range(BOOTSTRAP_B))]
        boot[m] = {"macro_f1_mean": float(np.mean(vals)),
                   "macro_f1_ci95": [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]}
    RESULTS["stats"] = {"mcnemar_raw": raw, "mcnemar_holm": holm, "bootstrap": boot}
    save_results(); del preds; gc.collect()

sig = {k: v for k, v in RESULTS["stats"]["mcnemar_holm"].items() if v < .05}
print(f"paires significativement differentes (Holm < 0,05) : {len(sig)}/{len(RESULTS['stats']['mcnemar_holm'])}")
bt = RESULTS["stats"]["bootstrap"]; ms = sorted(bt, key=lambda m: bt[m]["macro_f1_mean"])
mu = [bt[m]["macro_f1_mean"] for m in ms]
lo = [mu[i] - bt[m]["macro_f1_ci95"][0] for i, m in enumerate(ms)]
hi = [bt[m]["macro_f1_ci95"][1] - mu[i] for i, m in enumerate(ms)]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.errorbar(mu, range(len(ms)), xerr=[lo, hi], fmt="o", ms=4, capsize=3, color="#4C72B0")
ax.set_yticks(range(len(ms))); ax.set_yticklabels(ms)
ax.set_xlabel("macro-F1 (IC 95 % bootstrap)"); ax.set_title("Figure 7 — classement avec IC")
plt.savefig(FIGS / "fig7_ranking_ci.png"); plt.savefig(FIGS / "fig7_ranking_ci.pdf"); plt.show()

best = max((m for m in ALL_MODELS if m != "majority"),
           key=lambda m: (agg(m, "audited")[0] if not np.isnan(agg(m, "audited")[0]) else -1))
print("meilleur modele :", best)
pte = load_probs(f"{best}|audited|strat_seed1")[1]
cm = confusion_matrix(yte_ref, pte.argmax(1), labels=range(C), normalize="true")
fig, ax = plt.subplots(figsize=(5.8, 4.9))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(C)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(C)); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
for i in range(C):
    for j in range(C):
        if cm[i, j] >= .01:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if cm[i, j] > .5 else "black")
ax.set_xlabel("classe predite"); ax.set_ylabel("classe reelle")
ax.set_title(f"Figure 6 — matrice de confusion, {best}")
plt.colorbar(im); plt.savefig(FIGS / "fig6_confusion.png"); plt.savefig(FIGS / "fig6_confusion.pdf"); plt.show()
print(classification_report(yte_ref, pte.argmax(1), target_names=CLASS_NAMES, digits=3, zero_division=0))


In [ ]:
# 6.7 — Figures d'annexe pour le rapport de these (A7 a A15)
avail = [m for m in ALL_MODELS if done(f"{m}|audited|strat_seed1")]

# A7 courbes d'apprentissage
hk = [f"{m}|audited|strat_seed1" for m in DEEP_NAMES if f"{m}|audited|strat_seed1" in RESULTS["history"]]
if hk:
    fig, axes = plt.subplots(2, len(hk), figsize=(3.1 * len(hk), 5.4), squeeze=False)
    for j, k in enumerate(hk):
        h = RESULTS["history"][k]; nm = k.split("|")[0]
        axes[0][j].plot(h.get("accuracy", []), label="train")
        axes[0][j].plot(h.get("val_accuracy", []), label="validation")
        axes[0][j].set_title(f"{nm} — accuracy", fontsize=9); axes[0][j].set_xlabel("epoque")
        axes[1][j].plot(h.get("loss", []), label="train")
        axes[1][j].plot(h.get("val_loss", []), label="validation")
        axes[1][j].set_title(f"{nm} — perte", fontsize=9); axes[1][j].set_xlabel("epoque")
    axes[0][0].legend(fontsize=7); axes[1][0].legend(fontsize=7)
    fig.suptitle("A7 — courbes d'apprentissage")
    plt.savefig(SUPP / "A7_learning_curves.png"); plt.savefig(SUPP / "A7_learning_curves.pdf"); plt.show()

# A8 matrices de confusion de tous les modeles
ncol = 4; nrow = int(np.ceil(len(avail) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow))
for ax, m in zip(np.atleast_1d(axes).flat, avail):
    p = load_probs(f"{m}|audited|strat_seed1")[1]
    cmm = confusion_matrix(yte_ref, p.argmax(1), labels=range(C), normalize="true")
    ax.imshow(cmm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{m} (mF1 {one(m, 'audited', 'strat_seed1'):.3f})", fontsize=8)
    ax.set_xticks(range(C)); ax.set_yticks(range(C))
    ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=5); ax.set_yticklabels(CLASS_NAMES, fontsize=5)
for ax in np.atleast_1d(axes).flat[len(avail):]: ax.axis("off")
fig.suptitle("A8 — matrices de confusion, tous les modeles")
plt.savefig(SUPP / "A8_confusion_all.png"); plt.savefig(SUPP / "A8_confusion_all.pdf"); plt.show()

# A9 ROC et precision-rappel
fig, axes = plt.subplots(1, 2, figsize=(10, 4)); is_att = yte_ref != BENIGN_IDX
for m in avail:
    if m == "majority": continue
    p = load_probs(f"{m}|audited|strat_seed1")[1]; sc = 1.0 - p[:, BENIGN_IDX]
    fpr_, tpr_, _ = roc_curve(is_att, sc); pr_, rc_, _ = precision_recall_curve(is_att, sc)
    axes[0].plot(fpr_, tpr_, lw=1.2, label=f"{m} ({roc_auc_score(is_att, sc):.4f})")
    axes[1].plot(rc_, pr_, lw=1.2, label=f"{m} ({average_precision_score(is_att, sc):.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.6)
axes[0].set_xlabel("taux de faux positifs"); axes[0].set_ylabel("taux de vrais positifs")
axes[0].set_title("ROC — benin vs attaque"); axes[0].legend(fontsize=6, loc="lower right")
axes[1].set_xlabel("rappel"); axes[1].set_ylabel("precision")
axes[1].set_title("Precision-rappel"); axes[1].legend(fontsize=6, loc="lower left")
fig.suptitle("A9 — courbes ROC et precision-rappel")
plt.savefig(SUPP / "A9_roc_pr.png"); plt.savefig(SUPP / "A9_roc_pr.pdf"); plt.show()

# A10 F1 par classe / A11 temps d'entrainement
pcf = pd.DataFrame({m: RESULTS["models"][f"{m}|audited|strat_seed1"]["per_class_f1"]
                    for m in avail}).reindex(CLASS_NAMES)
display(pcf.round(4))
fig, ax = plt.subplots(figsize=(10, 4)); pcf.plot.bar(ax=ax, width=.85)
ax.set_ylabel("F1"); ax.set_ylim(0, 1.02); ax.set_title("A10 — F1 par classe et par modele")
ax.legend(fontsize=6, ncol=6)
plt.savefig(SUPP / "A10_per_class_f1.png"); plt.savefig(SUPP / "A10_per_class_f1.pdf"); plt.show()

ft = {m: RESULTS["models"][f"{m}|audited|strat_seed1"]["fit_time_s"] for m in avail}
fig, ax = plt.subplots(figsize=(6.5, 3.4)); ks = sorted(ft, key=ft.get)
ax.barh(ks, [max(ft[k], 1e-3) for k in ks], color="#8172B2"); ax.set_xscale("log")
ax.set_xlabel("temps d'entrainement (s, log)"); ax.set_title("A11 — cout d'entrainement")
plt.savefig(SUPP / "A11_train_time.png"); plt.savefig(SUPP / "A11_train_time.pdf"); plt.show()

# A12 gain du reglage / A13 tirages
if "hpo" in RESULTS:
    ms_ = list(RESULTS["hpo"].keys()); xs = np.arange(len(ms_)); w = .38
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.bar(xs - w/2, [agg(m, "audited")[0] for m in ms_], w, label="defaut", color="#4C72B0")
    ax.bar(xs + w/2, [agg(m + "#tuned", "audited")[0] for m in ms_], w, label="regle", color="#DD8452")
    ax.set_xticks(xs); ax.set_xticklabels(ms_, rotation=30, ha="right")
    ax.set_ylabel("macro-F1 (audite, 5 graines)"); ax.legend(fontsize=8)
    ax.set_title("A12 — effet du reglage d'hyperparametres")
    plt.savefig(SUPP / "A12_tuning_gain.png"); plt.savefig(SUPP / "A12_tuning_gain.pdf"); plt.show()

    ncol = 4; nrow = int(np.ceil(len(ms_) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 2.5 * nrow))
    for ax, m in zip(np.atleast_1d(axes).flat, ms_):
        v = [t["val_macro_f1"] for t in RESULTS["hpo"][m]["trials"] if t["val_macro_f1"] is not None]
        ax.plot(range(len(v)), v, "o", ms=3, color="#4C72B0")
        ax.axhline(RESULTS["hpo"][m]["best_val_macro_f1"], ls="--", c="#C44E52", lw=.8)
        ax.set_title(m, fontsize=8); ax.set_xlabel("tirage", fontsize=7)
    for ax in np.atleast_1d(axes).flat[len(ms_):]: ax.axis("off")
    fig.suptitle("A13 — tirages de la recherche aleatoire (validation)")
    plt.savefig(SUPP / "A13_hpo_trials.png"); plt.savefig(SUPP / "A13_hpo_trials.pdf"); plt.show()

# A14 autoencodeur / A15 significativite
p_ae = SAVE / "ae_scores_seed1.npz"
if p_ae.exists():
    z = np.load(p_ae); err, yv_ = z["err"], z["y"]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    bp = ax.boxplot([np.log10(err[yv_ == i] + 1e-12) for i in range(C)],
                    labels=CLASS_NAMES, showfliers=False, patch_artist=True)
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("#55A868" if i == BENIGN_IDX else "#C44E52"); b.set_alpha(.75)
    ax.set_ylabel("log10 erreur de reconstruction")
    ax.set_title("A14 — separation par l'autoencodeur (entraine sur le benin seul)")
    plt.xticks(rotation=30, ha="right")
    plt.savefig(SUPP / "A14_ae_scores.png"); plt.savefig(SUPP / "A14_ae_scores.pdf"); plt.show()

if "stats" in RESULTS:
    ms_ = sorted({k.split("|")[0] for k in RESULTS["stats"]["mcnemar_holm"]} |
                 {k.split("|")[1] for k in RESULTS["stats"]["mcnemar_holm"]})
    M = np.full((len(ms_), len(ms_)), np.nan)
    for k, v in RESULTS["stats"]["mcnemar_holm"].items():
        a, b = k.split("|"); i, j = ms_.index(a), ms_.index(b); M[i, j] = M[j, i] = v
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    im = ax.imshow(M, cmap="viridis_r", vmin=0, vmax=.1)
    ax.set_xticks(range(len(ms_))); ax.set_xticklabels(ms_, rotation=90, fontsize=7)
    ax.set_yticks(range(len(ms_))); ax.set_yticklabels(ms_, fontsize=7)
    plt.colorbar(im, label="p (McNemar, Holm)")
    ax.set_title("A15 — significativite des differences")
    plt.savefig(SUPP / "A15_mcnemar.png"); plt.savefig(SUPP / "A15_mcnemar.pdf"); plt.show()
gc.collect()


## 7. Sauvegarde des modèles et export

In [ ]:
# 7.1 — Bundle de deploiement
if SAVE_MODELS:
    _, _, sc_a = make_xy(F_AUDIT, "strat_seed1")
    _, _, sc_c = make_xy(F_CLEAN, "strat_seed1")
    (MODELS / "preprocessing.json").write_text(json.dumps({
        "scaler": "RobustScaler ajuste sur le train de strat_seed1",
        "audited": {"features": F_AUDIT, "center_": sc_a.center_.tolist(), "scale_": sc_a.scale_.tolist()},
        "clean":   {"features": F_CLEAN, "center_": sc_c.center_.tolist(), "scale_": sc_c.scale_.tolist()},
        "class_names": CLASS_NAMES, "benign_index": BENIGN_IDX, "blacklist": BLACKLIST,
        "identifiers_excluded": IDENTIFIERS, "positional": POSITIONAL}, indent=1), encoding="utf-8")
    shutil.copy(SPLITS_PATH, MODELS / "frozen_splits_60s.npz")
    (MODELS / "MANIFEST.json").write_text(json.dumps({
        "paper": "A Leakage-Audited Benchmark of Deep and Ensemble Detectors on the GeNIS 2025 Corpus",
        "corpus": "GeNIS 2025, 2-flows, 60 s, 9 classes (doi:10.5281/zenodo.14919237)",
        "reference_config": "condition auditee, split strat_seed1", "arms": ["default", "tuned"],
        "knn_max_train": KNN_MAX_TRAIN,
        "calibration_temperatures": {k: v["T"] for k, v in RESULTS.get("calibration", {}).items()},
        "files": sorted(p.name for p in MODELS.iterdir()),
        "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=1), encoding="utf-8")
    tot = sum(p.stat().st_size for p in MODELS.iterdir()) / 1e6
    print(f"{len(list(MODELS.iterdir()))} fichiers dans models/ ({tot:.1f} Mo)")
    for p in sorted(MODELS.iterdir()): print("  -", p.name)
    del sc_a, sc_c; gc.collect()


In [ ]:
# 7.2 — Archives, chiffres cles, telechargement
save_results()
exp = pathlib.Path("/content/export"); shutil.rmtree(exp, ignore_errors=True); exp.mkdir()
shutil.copytree(FIGS, exp / "figures"); shutil.copytree(SUPP, exp / "figures_annexe")
shutil.copytree(TABS, exp / "tables"); shutil.copy(RES_PATH, exp / "article1_results.json")
shutil.make_archive("/content/article1_figures", "zip", exp)
print(f"figures + tables : {os.path.getsize('/content/article1_figures.zip')/1e6:.1f} Mo")
if SAVE_MODELS:
    shutil.make_archive(str(SAVE / "article1_models"), "zip", MODELS)
    print(f"modeles : {os.path.getsize(SAVE/'article1_models.zip')/1e6:.1f} Mo "
          f"-> {SAVE/'article1_models.zip'} (conserve sur Drive)")

print("\n" + "=" * 72); print("CHIFFRES CLES"); print("=" * 72)
pr = RESULTS["shortcut_probes"]
print(f"corpus 60 s          : {RESULTS['slice60']['n']:,} flux | {C} classes | "
      f"benin {RESULTS['slice60']['benign_share']:.1%}")
print(f"sonde StartTime seul : stratifie {pr['starttime_only']['strat_mean']:.4f} | "
      f"temporel {pr['starttime_only']['temporal']:.4f} | hasard {pr['chance_majority']:.4f}")
print(f"liste noire ({len(BLACKLIST)})     : {BLACKLIST}")
print(f"features conservees  : {len(F_AUDIT)} / {len(F_FULL)}")
mu, sd = agg(best, "audited")
print(f"meilleur modele      : {best} — macro-F1 {mu:.4f} +/- {sd:.4f} (stratifie) | "
      f"{one(best, 'audited', 'temporal'):.4f} (temporel)")
if "autoencoder" in RESULTS:
    g = [v["auroc_global"] for v in RESULTS["autoencoder"].values()]
    print(f"autoencodeur         : AUROC {np.mean(g):.4f} +/- {np.std(g):.4f}")
print("=" * 72)
_ = inventory()

from google.colab import files
files.download("/content/article1_figures.zip")
files.download(str(RES_PATH))
print("\nRenvoyer article1_results.json et article1_figures.zip.")


---
## Aide-mémoire

**Après une déconnexion.** Exécutez §1.1 → §1.4 (environ 2 min grâce au cache), lisez
l'inventaire, puis relancez uniquement la cellule §5 correspondante. Rien n'est recalculé
deux fois : `run_batch` saute tout ce qui figure déjà sur Drive.

**Si la RAM sature malgré tout.** Dans l'ordre : (1) *Exécution → Redémarrer la session*,
puis §1 et un seul lot §5 ; (2) passez `RUN_INTERVALS = False` pour cette session ;
(3) dans §5.3b / §5.5b, réduisez le lot en éditant le filtre, par exemple
`r[0] in ["dnn", "cnn"]` puis `r[0] in ["rnn", "ftt"]`.

**Sans GPU.** Faites les lots rapides (§5.1a, §5.3a, §5.5a) : ils suffisent à produire le
tableau principal des modèles d'arbres. Gardez les lots profonds pour une session où le GPU
est disponible.

**Livrables.** `figures/` Fig. 1–8 (article) · `figures_annexe/` A1–A15 (rapport de thèse) ·
`tables/` T1–T4 (LaTeX) · `models/` modèles des deux bras, autoencodeur, scaler, liste noire,
splits gelés, manifeste.

**Points de vigilance à me signaler.** Sonde `StartTime` inférieure à 0,95 en stratifié ·
un modèle à macro-F1 ≥ 0,999 en condition auditée **et** en temporel · une classe absente
d'une partition du split temporel (l'invariant §3.1 doit afficher `OK`).
